In [ ]:
# @title Run Daily Options Screener (Earnings Filter + TradingView Output)
# 1. INSTALL LIBRARIES
!pip install yfinance pandas_ta -q

import yfinance as yf
import pandas as pd
import pandas_ta as ta
import numpy as np
from datetime import datetime, timedelta
import pytz

# 2. DEFINE THE TICKER LIST
raw_tickers = """
A AA AAL AAP AAPL ABBV ABNB ABT ACN ADBE ADI ADM ADP ADSK AEP AES AFL AFRM AGNC AI AIG AKAM ALB ALK ALL AMAT AMD AMGN AMRN AMZN APA APH APTV AVGO AXP BA BABA BAC BAX BBY BEN BIDU BIIB BITO BK BKR BMY BP BSX BX BYND C CAH CAT CB CCI CCJ CCL CF CFG CHWY CI CL CLF CLX CMCSA CME CNC CNP COF COIN COP COST CPB CPRI CRM CRON CRWD CSCO CSX CTVA CVNA CVS CVX CZR D DAL DD DE DELL DHI DHR DIA DIS DKNG DLR DOCU DOW DRI DVN DXC EA EBAY ED EEM EFA EIX EL EMR ENPH EOG EQR EQT ETSY EVRG EW EWJ EWW EWY EWZ EXC EXPE F FANG FAST FCX FDX FE FI FITB FOXA FSLR FTI FTV FXE FXI GD GDX GE GILD GLD GLW GM GME GOOG GOOGL GPRO GPS GS HAL HBAN HBI HCA HD HIG HLT HOG HOLX HON HPE HPQ HYG IBB IBIT IBM ICE IEF INTC IP IPG IRM IVZ IWM IYR JCI JD JETS JNJ JNPR JPM K KHC KMI KO KR KRE KSS KWEB LEN LI LKQ LLY LNC LOW LQD LUMN LUV LVS LYB LYFT MA MAR MARA MAS MCD MCHP MDLZ MDT MET META MGM MMM MO MOS MPC MRK MRNA MRVL MS MSFT MSTR MTB MTCH MU NCLH NEE NEM NET NFLX NKE NOW NRG NTAP NTES NVAX NVDA NWL NWSA O OIH OKE OMC ON ORCL OXY PARA PBR PDD PENN PEP PFE PFG PG PGR PINS PLTR PNC PPL PRU PSX PTON PYPL QCOM QQQ RBLX RCL RF RIG RIOT RIVN ROKU ROST RTX RUN SBUX SCHD SCHW SEDG SHOP SIRI SLB SLV SMCI SMH SNAP SNOW SO SOFI SOXL SOXS SPG SPX SPXL SPXS SPY SQQQ STX SWKS SYF SYY T TAP TBT TCOM TDOC TFC TGT TJX TLT TMO TMUS TPR TQQQ TRIP TRV TSLA TSM TSN TT TTD TTWO TXN U UA UAA UAL UBER ULTA UNG UNH UNP UPS UPST URBN USB USO UVXY V VALE VFC VGK VTR VXX VZ WAB WBA WBD WDC WFC WM WMB WMT WU WY WYNN X XBI XEL XHB XLB XLC XLE XLF XLI XLK XLP XLRE XLU XLV XLY XOM XOP XRT XRX XSP XYZ YELP YINN ZION ZM
"""
ticker_list = [t.strip() for t in raw_tickers.split() if t.strip() and t.strip() != "XYZ"]

# 3. HELPER FUNCTION: EARNINGS CHECK
def has_earnings_coming_up(ticker_symbol, days=9):
    """
    Returns True if earnings are within the next 'days' days.
    """
    try:
        t = yf.Ticker(ticker_symbol)
        # Get earnings dates
        cal = t.calendar
        if cal is None or len(cal) == 0:
            return False # Assume safe if no data (common with ETFs)

        # Handling different yfinance return structures
        if isinstance(cal, dict) and 'Earnings Date' in cal:
             earnings_dates = cal['Earnings Date']
        elif isinstance(cal, pd.DataFrame):
            # Often yfinance returns a dataframe with columns 0, 1 etc for dates
            # We look for the first valid date row
            return False # Fallback for now to avoid breaking on dataframe complexity
        else:
            return False

        # If we found a list of dates
        for date in earnings_dates:
             # Make timezone naive for comparison
             if date.tzinfo:
                 date = date.tz_localize(None)
             
             now = datetime.now()
             delta = date - now
             
             if 0 <= delta.days <= days:
                 return True
        return False
    except:
        return False # Fail safe: assume no earnings if API errors

# 4. DOWNLOAD DATA & FILTER LIQUIDITY
print(f"1/4: Downloading data for {len(ticker_list)} tickers...")
data = yf.download(ticker_list, period="6mo", group_by='ticker', auto_adjust=True, threads=True, progress=False)

print("2/4: Filtering Top 200 by Liquidity (Dollar Volume)...")
liquidity_stats = []
for ticker in ticker_list:
    try:
        if ticker not in data.columns.levels[0]: continue
        df = data[ticker].dropna()
        if df.empty: continue
        
        avg_price = df['Close'].tail(20).mean()
        avg_vol = df['Volume'].tail(20).mean()
        liquidity_stats.append({'Ticker': ticker, 'DollarVolume': avg_price * avg_vol})
    except: continue

liq_df = pd.DataFrame(liquidity_stats).sort_values(by='DollarVolume', ascending=False).head(200)
top_200_tickers = liq_df['Ticker'].tolist()

# 5. APPLY STRATEGY & EARNINGS FILTER
print("3/4: Screening Technicals & Checking Earnings (This may take a moment)...")
candidates = []

for ticker in top_200_tickers:
    try:
        df = data[ticker].dropna().copy()
        if len(df) < 50: continue

        # Indicators
        df['SMA_10'] = ta.sma(df['Close'], length=10)
        df['SMA_50'] = ta.sma(df['Close'], length=50)
        df['RSI'] = ta.rsi(df['Close'], length=14)
        current = df.iloc[-1]
        
        signal = None
        
        # Bullish Criteria
        if (current['Close'] > current['SMA_50']) and \
           (current['Close'] > current['SMA_10']) and \
           (40 < current['RSI'] < 65):
            signal = "BULLISH"

        # Bearish Criteria
        elif (current['Close'] < current['SMA_50']) and \
             (current['Close'] < current['SMA_10']) and \
             (35 < current['RSI'] < 60):
            signal = "BEARISH"
            
        if signal:
            # ONLY Check earnings if technicals pass (saves API calls)
            if not has_earnings_coming_up(ticker, days=9):
                candidates.append({
                    'Ticker': ticker,
                    'Signal': signal,
                    'DollarVolume': liq_df[liq_df['Ticker'] == ticker]['DollarVolume'].values[0],
                    'Close': current['Close'],
                    'SMA_10': current['SMA_10']
                })
    except: continue

# 6. SELECT BEST 30 (15 MAX PER SIDE)
results_df = pd.DataFrame(candidates)
final_list = []

if not results_df.empty:
    # Split by signal
    bulls = results_df[results_df['Signal'] == 'BULLISH'].sort_values(by='DollarVolume', ascending=False).head(15)
    bears = results_df[results_df['Signal'] == 'BEARISH'].sort_values(by='DollarVolume', ascending=False).head(15)
    
    # Create Lists
    bull_tickers = bulls['Ticker'].tolist()
    bear_tickers = bears['Ticker'].tolist()
    
    print("\n" + "="*80)
    print("TRADINGVIEW WATCHLIST STRINGS (COPY & PASTE)")
    print("="*80)
    
    print("\n>>> BULLISH WATCHLIST (Sell Puts):")
    print(", ".join(bull_tickers))
    
    print("\n>>> BEARISH WATCHLIST (Sell Calls):")
    print(", ".join(bear_tickers))
    
    print("\n" + "="*80)
    print("DETAILED RESULTS")
    print("="*80)
    print(pd.concat([bulls, bears])[['Ticker', 'Signal', 'Close', 'SMA_10']].to_string(index=False))

else:
    print("No trades found matching criteria today.")

# 7. PRINT RULES
print("\n" + "="*80)
print("⚠️ TRADE MANAGEMENT RULES (MUST FOLLOW)")
print("="*80)
print("1. ENTRY:")
print("   - Expiration: ~9 Days to Expiration (DTE).")
print("   - Strike: Sell strike with ~$0.20 - $0.25 credit (approx 15-20 Delta).")
print("   - Spread Width: $1.00 (or $2.50/5.00 for larger stocks like SPY/AMZN).")
print("\n2. STOP LOSS (THE EDGE):")
print("   - Set a price alert at your SOLD STRIKE.")
print("   - IF Stock Price CROSSES/CLOSES past your Sold Strike -> CLOSE THE TRADE.")
print("   - DO NOT HOLD hoping for a reversal. Take the small loss (~$25-$30).")
print("\n3. TAKE PROFIT:")
print("   - Option A: Let expire worthless (100% profit) if trend holds.")
print("   - Option B: Close at 50% profit to free up capital.")
print("="*80)